In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from statsmodels.stats.contingency_tables import mcnemar
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading data...")
train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")


tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()
label_encoder = LabelEncoder()
label_encoder.fit(tag_vocab)

train_data["EncodedTags"] = label_encoder.transform(train_data["language"])
test_data["EncodedTags"] = label_encoder.transform(test_data["language"])

train_df = train_data.rename(columns={'code': 'input_text', 'EncodedTags': 'target_text'})
test_df = test_data.rename(columns={'code': 'input_text', 'EncodedTags': 'target_text'})

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Number of unique languages: {len(tag_vocab)}")

In [ ]:
TOKENIZATION_PATTERNS = {
    # Baseline 1: Simple whitespace tokenization
    "whitespace_baseline": r'\S+',    
    # Our Pattern
    "our_full_pattern": r'(\b[A-Za-z_]\w*\b|[!\#\$%\&\*\+:\-\./<=>\?@\\\^_\|\~]+|[ \t\(\),;\{\}\[\]`"\'])',
    
    # Component analysis: Individual components
    "identifiers_only": r'\b[A-Za-z_]\w*\b',
    
    "operators_only": r'[!\#\$%\&\*\+:\-\./<=>\?@\\\^_\|\~]+',
    
    "separators_only": r'[ \t\(\),;\{\}\[\]`"\']',
    
    # Two-component combinations
    "identifiers_operators": r'\b[A-Za-z_]\w*\b|[!\#\$%\&\*\+:\-\./<=>\?@\\\^_\|\~]+',
    
    "identifiers_separators": r'\b[A-Za-z_]\w*\b|[ \t\(\),;\{\}\[\]`"\']',
    
    }

In [ ]:
def preprocess(x):
    """Remove single letters and repeated letters from code"""
    return pd.Series(x).replace(r'\b([A-Za-z])\1+\b', '', regex=True)\
        .replace(r'\b[A-Za-z]\b', '', regex=True)

In [ ]:
def run_ablation_study(train_df, test_df, patterns_dict, selected_patterns=None):
    """
    Run ablation study comparing different tokenization patterns
    
    Parameters:
    -----------
    train_df, test_df: DataFrames with 'input_text' and 'target_text' columns
    patterns_dict: Dictionary of pattern names and regex patterns
    selected_patterns: List of specific patterns to test (if None, test all)
    
    Returns:
    --------
    DataFrame with results for each pattern
    """
    
    results = []
    all_predictions = {}
    
    if selected_patterns:
        patterns_to_test = {k: v for k, v in patterns_dict.items() if k in selected_patterns}
    else:
        patterns_to_test = patterns_dict
    
    print(f"\nTesting {len(patterns_to_test)} tokenization patterns...")
    
    for pattern_name, pattern in patterns_to_test.items():
        print(f"\n{'='*60}")
        print(f"Testing pattern: {pattern_name}")
        print(f"Pattern: {pattern[:80]}..." if len(pattern) > 80 else f"Pattern: {pattern}")
        print(f"{'='*60}")
        
        vectorizer = TfidfVectorizer(
            token_pattern=pattern,
            max_features=10000,
            lowercase=False,  
            min_df=2,  
            max_df=0.9  
        )
        
        base_estimator = RandomForestClassifier(
            n_jobs=4,
            random_state=42 
        )
        
        pipe = Pipeline([
            ('preprocessing', FunctionTransformer(preprocess, validate=False)),
            ('vectorizer', vectorizer),
            ('clf', OneVsRestClassifier(base_estimator))
        ])
        
        best_params = {
            'clf__estimator__criterion': 'gini',
            'clf__estimator__max_features': 'log2',
            'clf__estimator__min_samples_split': 3,
            'clf__estimator__n_estimators': 400
        }
        pipe.set_params(**best_params)
        
        # Train
        print("Training model...")
        pipe.fit(train_df.input_text, train_df.target_text)
        
        # Test
        print("Evaluating...")
        predictions = pipe.predict(test_df.input_text)
        all_predictions[pattern_name] = predictions
        
        accuracy = accuracy_score(test_df.target_text, predictions)
    
        report = classification_report(
            test_df.target_text, 
            predictions, 
            output_dict=True, 
            zero_division=0
        )
        
        # Extract key metrics
        macro_f1 = report['macro avg']['f1-score']
        weighted_f1 = report['weighted avg']['f1-score']
        
        if pattern_name in ["our_full_pattern", "whitespace_baseline", "with_numbers"]:
            print(f"\nTop languages by F1-score for {pattern_name}:")
            for lang_idx in range(len(label_encoder.classes_)):
                if str(lang_idx) in report:
                    lang_name = label_encoder.inverse_transform([lang_idx])[0]
                    f1 = report[str(lang_idx)]['f1-score']
                    if f1 > 0.8:  # Show only high-performing languages
                        print(f"  {lang_name}: F1 = {f1:.3f}")
        
        # Analyze token statistics for the first training sample
        if pattern_name in ["our_full_pattern", "whitespace_baseline"]:
            try:
                analyzer = vectorizer.build_analyzer()
                sample_text = train_df.input_text.iloc[0]
                tokens = analyzer(sample_text)
                print(f"\nToken analysis for first training sample:")
                print(f"  Total tokens: {len(tokens)}")
                print(f"  Unique tokens: {len(set(tokens))}")
                print(f"  Sample tokens: {list(tokens)[:15]}...")
            except:
                pass
        
        # Store results
        results.append({
            'pattern_name': pattern_name,
            'pattern': pattern,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'model': pipe,
            'predictions': predictions
        })
        
        print(f"\nResults - Accuracy: {accuracy:.4f}, Macro-F1: {macro_f1:.4f}, Weighted-F1: {weighted_f1:.4f}")
    
    results_df = pd.DataFrame(results)
    return results_df, all_predictions

In [ ]:
def visualize_results(results_df):
    """Create visualizations of the ablation study results"""
    
    results_df_sorted = results_df.sort_values('accuracy', ascending=False)
    
    # Plot 1: Bar chart of accuracies
    plt.figure(figsize=(14, 8))
    bars = plt.barh(range(len(results_df_sorted)), results_df_sorted['accuracy'])
    plt.yticks(range(len(results_df_sorted)), results_df_sorted['pattern_name'])
    plt.xlabel('Accuracy')
    plt.title('Ablation Study: Tokenization Patterns for Code Classification')
    plt.xlim([results_df_sorted['accuracy'].min() - 0.02, 
              results_df_sorted['accuracy'].max() + 0.02])

    for i, (bar, pattern_name) in enumerate(zip(bars, results_df_sorted['pattern_name'])):
        if 'baseline' in pattern_name:
            bar.set_color('red')
        elif 'only' in pattern_name:
            bar.set_color('orange')
        elif pattern_name == 'our_full_pattern':
            bar.set_color('green')

            plt.text(results_df_sorted['accuracy'].iloc[i] + 0.002, i, 
                    f"Best: {results_df_sorted['accuracy'].iloc[i]:.4f}", 
                    va='center', fontweight='bold')
        elif pattern_name.startswith('with_'):
            bar.set_color('blue')
        elif 'operators' in pattern_name and 'identifiers' in pattern_name:
            bar.set_color('purple')
    
    plt.tight_layout()
    plt.savefig('ablation_study_accuracy.png', dpi=300, bbox_inches='tight')
    
    # Plot 2: Comparison of all metrics
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    metrics = ['accuracy', 'macro_f1', 'weighted_f1']
    titles = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx]
        sorted_idx = np.argsort(results_df[metric])[::-1]
        y_pos = np.arange(len(results_df))
        
        bars = ax.barh(y_pos, results_df[metric].iloc[sorted_idx])
        ax.set_yticks(y_pos)
        ax.set_yticklabels(results_df['pattern_name'].iloc[sorted_idx])
        ax.set_xlabel(title)
        ax.set_title(f'{title} by Pattern')
        
        for i, pattern_name in enumerate(results_df['pattern_name'].iloc[sorted_idx]):
            if pattern_name == 'our_full_pattern':
                bars[i].set_color('green')
                ax.text(results_df[metric].iloc[sorted_idx[i]] + 0.002, i, 
                       f"{results_df[metric].iloc[sorted_idx[i]]:.4f}", 
                       va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('ablation_study_all_metrics.png', dpi=300, bbox_inches='tight')
    
    return results_df_sorted


In [ ]:
def perform_statistical_tests(results_df, test_df, all_predictions):
    """Perform statistical significance tests"""
    
    print("\n" + "="*80)
    print("STATISTICAL SIGNIFICANCE ANALYSIS")
    print("="*80)

    your_pattern_row = results_df[results_df['pattern_name'] == 'our_full_pattern']
    baseline_row = results_df[results_df['pattern_name'] == 'whitespace_baseline']
    
    if len(your_pattern_row) == 0 or len(baseline_row) == 0:
        print("Warning: Required patterns not found in results")
        return
    
    your_accuracy = your_pattern_row['accuracy'].values[0]
    baseline_accuracy = baseline_row['accuracy'].values[0]
    
    print(f"\nComparison: Your Pattern vs Baseline")
    print(f"  Your Pattern Accuracy:    {your_accuracy:.4f}")
    print(f"  Baseline Accuracy:        {baseline_accuracy:.4f}")
    print(f"  Absolute Improvement:     {your_accuracy - baseline_accuracy:.4f}")
    print(f"  Relative Error Reduction: {((your_accuracy - baseline_accuracy) / (1 - baseline_accuracy)):.2%}")
    
    # McNemar's test for paired nominal data
    your_preds = all_predictions['our_full_pattern']
    baseline_preds = all_predictions['whitespace_baseline']
    
    # Create contingency table
    table = np.zeros((2, 2))
    table[0, 0] = np.sum((your_preds == test_df.target_text) & (baseline_preds == test_df.target_text))
    table[0, 1] = np.sum((your_preds == test_df.target_text) & (baseline_preds != test_df.target_text))
    table[1, 0] = np.sum((your_preds != test_df.target_text) & (baseline_preds == test_df.target_text))
    table[1, 1] = np.sum((your_preds != test_df.target_text) & (baseline_preds != test_df.target_text))
    
    print(f"\nMcNemar's Test Contingency Table:")
    print(f"                   Baseline Correct   Baseline Wrong")
    print(f"Your Correct        {table[0,0]:10.0f}        {table[0,1]:10.0f}")
    print(f"Your Wrong          {table[1,0]:10.0f}        {table[1,1]:10.0f}")
    
    result = mcnemar(table, exact=True)
    print(f"\nMcNemar's test p-value: {result.pvalue:.6f}")
    
    if result.pvalue < 0.05:
        print("✓ Difference is statistically significant (p < 0.05)")
        significance = "significant"
    else:
        print("✗ Difference is not statistically significant")
        significance = "not significant"
    
    # Compare with other key patterns
    print("\n" + "-"*80)
    print("COMPARISON WITH OTHER PATTERNS")
    print("-"*80)
    
    key_patterns = ['identifiers_only', 'identifiers_operators', 'with_numbers']
    comparisons = []
    
    for pattern in key_patterns:
        if pattern in all_predictions:
            pattern_accuracy = results_df[results_df['pattern_name'] == pattern]['accuracy'].values[0]
            diff = your_accuracy - pattern_accuracy
            comparisons.append((pattern, pattern_accuracy, diff))
    
    comparisons.sort(key=lambda x: x[2], reverse=True)
    
    for pattern, acc, diff in comparisons:
        print(f"  vs {pattern:20s}: {acc:.4f} (Δ = {diff:+.4f})")
    
    return significance, table

In [ ]:
def analyze_token_categories(train_df, patterns_dict):
    """Analyze what types of tokens are captured by each pattern"""
    
    print("\n" + "="*80)
    print("TOKEN CATEGORY ANALYSIS")
    print("="*80)
    
    key_patterns = ['whitespace_baseline', 'identifiers_only', 'operators_only', 
                   'our_full_pattern', 'with_numbers']
    
    for pattern_name in key_patterns:
        pattern = patterns_dict[pattern_name]
        vectorizer = TfidfVectorizer(token_pattern=pattern, max_features=5000)
        
        sample_texts = train_df.input_text.head(1000)
        X = vectorizer.fit_transform(sample_texts)
        features = vectorizer.get_feature_names_out()
        
        identifier_count = sum(1 for f in features if re.match(r'^[A-Za-z_]', f))
        operator_count = sum(1 for f in features if re.match(r'^[^\w\s]', f))
        numeric_count = sum(1 for f in features if re.match(r'^\d', f))
        other_count = len(features) - identifier_count - operator_count - numeric_count
        
        print(f"\n{pattern_name}:")
        print(f"  Total unique tokens: {len(features)}")
        print(f"  Identifiers: {identifier_count} ({identifier_count/len(features)*100:.1f}%)")
        print(f"  Operators: {operator_count} ({operator_count/len(features)*100:.1f}%)")
        print(f"  Numbers: {numeric_count} ({numeric_count/len(features)*100:.1f}%)")
        print(f"  Other: {other_count} ({other_count/len(features)*100:.1f}%)")
        
        print(f"  Sample tokens: {features[:10]}")


In [ ]:
# Main execution
if __name__ == "__main__":
    print("Starting ablation study for CodeLite...")
    print(f"Training samples: {len(train_df)}")
    print(f"Test samples: {len(test_df)}")
    print(f"Number of languages: {len(tag_vocab)}")
    
    selected_patterns = [
        'whitespace_baseline',
        'identifiers_only',
        'operators_only',
        'separators_only',
        'identifiers_operators',
        'identifiers_separators',
        'our_full_pattern',
    ]
    
    results_df, all_predictions = run_ablation_study(
        train_df, 
        test_df, 
        TOKENIZATION_PATTERNS,
        selected_patterns=selected_patterns
    )
    
    results_sorted = visualize_results(results_df)
    
    significance, contingency_table = perform_statistical_tests(
        results_sorted, 
        test_df, 
        all_predictions
    )
    
    analyze_token_categories(train_df, TOKENIZATION_PATTERNS)
    
    print("\n" + "="*80)
    print("ABLATION STUDY COMPLETE - KEY FINDINGS")
    print("="*80)
    
    best_pattern = results_sorted.iloc[0]
    baseline = results_sorted[results_sorted['pattern_name'] == 'whitespace_baseline'].iloc[0]
    
    print(f"\n1. Best Pattern: {best_pattern['pattern_name']}")
    print(f"   Accuracy: {best_pattern['accuracy']:.4f}")
    print(f"   Improvement over baseline: {best_pattern['accuracy'] - baseline['accuracy']:.4f} "
          f"({((best_pattern['accuracy'] - baseline['accuracy']) / (1 - baseline['accuracy'])):.1%} relative)")
    
    print(f"\n2. Statistical significance: {significance}")
    
    print(f"\n3. Top 5 patterns:")
    for i in range(min(5, len(results_sorted))):
        row = results_sorted.iloc[i]
        print(f"   {i+1}. {row['pattern_name']}: {row['accuracy']:.4f}")
    
    print(f"\nResults saved to:")
    print("   - ablation_study_accuracy.png")
    print("   - ablation_study_all_metrics.png")
    print("   - ablation_study_results_detailed.csv")
    
    print(f"\n" + "="*80)
    print("SAMPLE PREDICTION COMPARISON")
    print("="*80)
    
    sample_indices = [0, 1, 2, 3, 4]
    for idx in sample_indices:
        actual_lang = label_encoder.inverse_transform([test_df.target_text.iloc[idx]])[0]
        print(f"\nSample {idx}: Actual = {actual_lang}")
        
        for pattern in ['whitespace_baseline', 'identifiers_only', 'our_full_pattern']:
            if pattern in all_predictions:
                pred_lang = label_encoder.inverse_transform([all_predictions[pattern][idx]])[0]
                print(f"  {pattern:25s}: {pred_lang}")